# Practical 05: ERP data analysis

In [16]:
# Credits: this notebook was written by Ilaria Gavetti for the Cognitive Neuroscience Executive Functions practical & EEGlab.

# MNE-toolbox was used as reference to develop and implement the code. 

! pip install mne

In [3]:
import mne
import os
import numpy as np
import csv
import pandas as pd

Welcome to the data-analysis part of the Executive Functions pracical. At this point, you should have:

1. collected your data 
2. received an output csv file from the data collection procedure, ... 


Follow the instructions below to properly upload and conduct your data-analysis:

# Step 0: upload the files

First, I suggest creating a "base" directory, called "Executive functions practical".

Then, please upload all the .csv files from your laptop in the directory **"/.../Executive functions practical/1. Raw files"**, where the **"..."** part is your personal path to get to **"Executive functions practical"** and **1. Raw files** is a folder in this directory which you should manually add.

Also, since we will be saving intermediate files during the analysis, add the following folders to your **"Executive functions practical"** directory:

**2. Loading & Digitization**

**3. ICA**

**4. Epoch** (with inside 1 other folder called **Annotated**)


The code you will run already has these directories implemented.

Last, but not least, change your current working directory to **"Executive functions practical"**, to ensure the code runs smoothly.

# Step 1: Loading

In [4]:
os.getcwd()

'/home/jovyan/NEW (22-23)/Student Assistant/CogNeuro re-design/Executive functions - ERP'

In [ ]:
# Read the CSV file as a NumPy array

for file in os.listdir("1. Raw"):
    if file != '.ipynb_checkpoints':
        if "EEG" in file:
            print(file)
            data = pd.read_csv("1. Raw/"+file, delimiter=",", verbose = True)
            data_sub = data[["FZ", "C3", "CZ", "C4", "PZ", "PO7", "OZ", "PO8"]] # only select the elecrodes columns, not the rest
            # Print file name
            print("****"+file+"****")
        elif "Unicorn" in file:
            print(file)
            data = pd.read_csv("1. Raw/"+file, delimiter=",", verbose = True)
            data_sub = data[["EEG 1", "EEG 2", "EEG 3", "EEG 4", "EEG 5", "EEG 6", "EEG 7", "EEG 8"]] # only select the elecrodes columns, not the rest
            # Print file name
            print("****"+file+"****")

        # Some information about the channels
        ch_names_new = ["Fz", "C3", "Cz", "C4", "Pz", "PO7", "Oz", "PO8"] # from Unicorn BCI manual

        # Sampling rate of the Unicorn BCI
        sfreq = 250  # Hz

        # Create the info structure needed by MNE
        info = mne.create_info(ch_names_new, sfreq, ch_types = ["eeg"]*len(ch_names_new), verbose = True)

        # Set montage       
        info.set_montage(montage = 'standard_1020')


        # Finally, create the Raw object
        raw = mne.io.RawArray(data_sub.T, info, verbose = True)

        # Filter 
        raw.filter(1,30)

        # Plot it!
        raw.plot(scalings="auto", duration = 5)
        print("End")

        # Save it!
        raw.save("2. Loading & Digitization/"+file[0:19]+"_raw.fif", overwrite = True)
        

# Step 2: Adding annotations manually

In [7]:
from datetime import datetime, timezone
from datetime import timedelta

In [8]:
os.getcwd()

'/home/jovyan/NEW (22-23)/Student Assistant/CogNeuro re-design/Executive functions - ERP'

# Add the name of the annotations file instead of '...' in all the cells below

In [10]:
np.load("2. Loading & Digitization/...")

array([['go', '1018'],
       ['go', '2149'],
       ['go', '3283'],
       ['no-go', '4417'],
       ['no-go', '5555'],
       ['go', '6693'],
       ['no-go', '7835'],
       ['go', '8975'],
       ['go', '10118'],
       ['go', '11261'],
       ['go', '12400'],
       ['go', '13548'],
       ['no-go', '14696'],
       ['no-go', '15841'],
       ['go', '16987'],
       ['go', '18135'],
       ['go', '19286'],
       ['go', '20438'],
       ['go', '21590'],
       ['no-go', '22747'],
       ['no-go', '23905'],
       ['go', '25059'],
       ['no-go', '26214'],
       ['go', '27369'],
       ['no-go', '28524'],
       ['go', '29683'],
       ['no-go', '30845'],
       ['go', '32003'],
       ['no-go', '33160'],
       ['no-go', '34320'],
       ['go', '35478'],
       ['go', '36634'],
       ['go', '37791'],
       ['go', '38952'],
       ['go', '40121'],
       ['go', '41286'],
       ['go', '42451'],
       ['go', '43629'],
       ['go', '44799'],
       ['no-go', '45967'],
       ['

Up until now, the raw data does not contain any markers, meaning that each file is a continuous flow of EEG data without any marker indicating where the non-oddball / odball trials happen.

Nevertheless, we do know at what time from the recording onset each trial happens, so we need to make sure we annotate the EEG data with markers that reflect the information given in the **array**.

For example, if the first line of the **array** says " *go* - *1000* ", this means that the *1000th* sample (an normal trial) was presented 4x1000 = 4000 **ms** after the recording onset, and so on. 

In the next cell, we make sure that the EEG data is annotated such that the program will know at what point of the contiinous EEG data each trial happens, and how long each trial lasts until the onset of a new trial.

Annotating will allow us to  "Epoch" the data, which will make our life easier later in the notebook!

In [11]:
files = os.listdir("2. Loading & Digitization")

In [ ]:
for file in files:
    if file != '.ipynb_checkpoints_raw.fif' and "raw" in file:
        if "EEG" in file:
            new_time = datetime(2023, 1, 27, hour=11, minute=59, second=20, microsecond=0, tzinfo=timezone.utc,fold=0) # form name of one of the two csv files
        elif "Unicorn" in file:
            new_time = datetime(2023, 1, 27, hour=11, minute=59, second=20, microsecond=0, tzinfo=timezone.utc,fold=0) # form name of one of the two csv files
        sample_data_raw_file = "2. Loading & Digitization/"+file
        raw = mne.io.read_raw_fif(sample_data_raw_file, preload= True)
        raw.set_meas_date(new_time)
        meas_date = raw.info["meas_date"]
        print(meas_date)

        raw.plot(scalings = "auto", duration = 5)

        # Annotation 
        array = np.load("2. Loading & Digitization/...") 

        description = []
        onset = []
        start = float(0)

        for i in range(array.shape[0]):
            description.append(array[i][0])
            onset.append(start)

            # Increase
            start = start + 4.5


        time_format = '%Y-%m-%d %H:%M:%S.%f'
        orig_time = (meas_date + timedelta(seconds=4)).strftime(time_format) # onset of annotations is 4 seconds after the start of the recording

        later_annot = mne.Annotations(onset=onset, duration=[0.5]*50, description=description, orig_time=orig_time)

        raw.set_annotations(later_annot)
        print(description)
        print("Annotations onsets:", raw.annotations.onset)
        print(meas_date, orig_time)


        raw.save("4. Epoch/Annotated files/"+file[0:19]+"_annot_raw.fif", overwrite = True)

# Step 3: epoching

In [14]:
from Epoching_erp import epoching

In [15]:
os.getcwd()

'/home/jovyan/NEW (22-23)/Student Assistant/CogNeuro re-design/Executive functions - ERP'

First of all, you're doing great! Congratulations for getting this far, we are almost there!

In the next step, epoching, you will "epoch" your files: this means that the program will take each file and extract the 0.5-seconds long trials marked by the markers we inserted in the Annotations step and save this information in a new, "Epoch" file, which can be thought of as a N-dimensional numpy array of shape: trials(the amount of markers we inserted in the Annotations step) X nr_channels X time points.

In this case, we also add a baseline period while epoching, in order to compare and correct for brain activity **before** and **after** the trial onset.

In [ ]:
for file in os.listdir("4. Epoch/Annotated files"):
    if "raw" in file and file != '.ipynb_checkpoints__annot_raw.fif':
        epoching(file, "4. Epoch/Annotated files")

# Load the two files you epoched to make sure the time ranges are correct: the files should report: t = -152.00....500ms >> from -0.152 to 0.5 seconds.

# Replace the '...' with the name of the epoched file

In [ ]:
epoch_one = mne.read_epochs("4. Epoch/...", preload= True)

**Same for this epoch:**

In [ ]:
epoch_one.get_data().shape

# Step 4: Event-related potential analysis (ERP)

The goal of ERP analysis is to plot across-trials averaged signal(s), where each plotted signal is the average of all the trials belonging to one specific class.  

To do so, the first step is to create two **evoked** objects (evoked objects are objects that result from the averaging across specified trials within an epoch) for each epoched file; each evoked file will extract and avergae across one of the two trial classess (go/no-go).

In [21]:
go_one = epoch_one['go'].average()
no_go_one = epoch_one['no-go'].average()

In the next step, we create a dictionary where the key is the trial type and the value is a list containing the corresponding evoked objects, for each epoched file.

In [22]:
evokeds = {"go": [go_one], "no-go": [no_go_one] }

Lastly, we plot the two ERP lines using a MNE function, to which we pass the following parameters: 
1. the "evokeds" dictionary
2. the method we want to use to combine the evoked objects
3. a color dictionary (color_dict) for the line plotting
4. a linestyle (linestyle_dict)
5. the time unit we want to visualise (ms are the standard for ERP plots)
6. the channels we would like to average across: since the P300 is known to peak in the posterior channels, we will pass such channels to this parameter, although feel free to tweak around!

In [ ]:
color_dict = {'go':'red', 'no-go':'blue'}
linestyle_dict = {'go':'-', 'no-go':'-'}
mne.viz.plot_compare_evokeds(evokeds, combine = "mean", colors=color_dict,
                             linestyles=linestyle_dict, time_unit ="ms")

You can also, for each trial class, visualize a topomap, in order to inspect the buildup of brain activity throughout the averaged epochs.

In [ ]:
mne.viz.plot_evoked_topomap((go_one), vlim = (-4500000, 4500000))
print("End")

In [ ]:
mne.viz.plot_evoked_topomap((no_go_one), vlim = (-4500000, 4500000));

# Step 5: Statistical analysis

In [32]:
from scipy import stats

In [ ]:
go_evoked = [go_one]
    
go_grand_av = mne.grand_average(go_evoked)

In [ ]:
no_go_evoked = [no_go_one]
    
no_go_grand_av = mne.grand_average(no_go_evoked)

In [33]:
stat, pvalue = stats.ttest_rel(go_grand_av.get_data(), no_go_grand_av.get_data())

In [ ]:
print(pvalue)

# The End!